In [1]:
import torch
from torch.optim import Adam
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from tqdm import tqdm
from sklearn.metrics import accuracy_score

2022-09-29 15:36:25.269083: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2022-09-29 15:36:26.174169: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2022-09-29 15:36:26.174260: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2022-09-29 15:36:26.174272: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make su

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

device

'cuda'

In [3]:
from datasets import Dataset
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizer-icd-final')
dataset = Dataset.load_from_disk('/root/data/mimic-iv_data/mimic_icd_hf_dataset')
dataset = dataset.train_test_split(test_size=0.1)

In [4]:
tokenization_params = {
    'max_length': 128,
    'truncation': True,
    'padding': 'max_length',
    'is_split_into_words': True,
    'return_special_tokens_mask': True
}

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizer-icd-final')

In [6]:
BATCH_SIZE = 64

LEARNING_RATE = 1e-4

EPOCHS = 5

DROPOUT_PROB = 0

In [7]:
import models

In [18]:
from models.bertha.modelling_bertha import BerthaForMaskedLM
from models.bertha.configuration_bertha import BerthaConfig

In [8]:
DROPOUT_PROB = 0

config = BerthaConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_dropout_prob=DROPOUT_PROB,
    attention_probs_dropout_prob=DROPOUT_PROB,
    max_position_embeddings=tokenization_params['max_length'],
    type_vocab_size=100,
    initializer_range=0.01
)

TypeError: 'module' object is not callable

In [9]:
model = Bertha.BertForMaskedLM(config)
model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(26148, 768, padding_idx=0)
      (position_embeddings): Embedding(128, 768)
      (token_type_embeddings): Embedding(100, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): BertEncoder(
      (embed_positions): RoFormerSinusoidalPositionalEmbedding(128, 64)
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self_attention): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=Tru

In [10]:
optimizer = Adam(params=model.parameters(), lr=LEARNING_RATE)

In [11]:
from transformers import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=6000, num_training_steps=30_000)

In [12]:
from transformers import Trainer
from transformers import TrainingArguments
import numpy as np
from datasets import load_metric
metric = load_metric('accuracy')

In [13]:
dataset['train'][0].keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'])

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    active_mask = labels.flatten() != -100
    active_labels = labels.flatten()[active_mask]
    active_preds = preds.flatten()[active_mask]
    return metric.compute(predictions=active_preds, references=active_labels)

In [15]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        logits = logits.permute(0, 2, 1)
        # compute custom loss (suppose one has 3 labels with different weights)
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)

        preds = torch.argmax(logits, axis=1)
        # Get active labels
        active_mask = labels.view(-1) != -100
        active_labels = torch.masked_select(labels.view(-1), active_mask)
        active_preds = torch.masked_select(preds.view(-1), active_mask)
        acc = metric.compute(predictions=active_preds.cpu().numpy(), references=active_labels.cpu().numpy())
        self.log(acc)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer_and_scheduler(self, num_training_steps: int):
        self.optimizer = Adam(params=self.model.parameters(), lr=self.args.learning_rate)
        self.lr_scheduler = get_cosine_schedule_with_warmup(optimizer=self.optimizer, num_warmup_steps=750, num_training_steps=num_training_steps)


In [16]:
training_args = TrainingArguments(
        output_dir=".",
        learning_rate=LEARNING_RATE,  # config
        do_train=True,
        do_eval=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        num_train_epochs=EPOCHS,  # config
        max_steps=-1,
        per_device_train_batch_size=BATCH_SIZE,  # config
        per_device_eval_batch_size=BATCH_SIZE,  # config
        logging_dir="./logs",
        gradient_accumulation_steps=1,
        fp16 = False,
        bf16 = False,
        tf32=False,
        ddp_find_unused_parameters=False,
    )

In [17]:
trainer = CustomTrainer(
    model=model,
    args = training_args,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer = tokenizer,
    # optimizers=(optimizer, lr_scheduler),
    data_collator = DataCollatorForLanguageModeling(tokenizer)
)

In [18]:
trainer.train()

The following columns in the training set don't have a corresponding argument in `BertForMaskedLM.forward` and have been ignored: special_tokens_mask. If special_tokens_mask are not expected by `BertForMaskedLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 171161
  Num Epochs = 5
  Instantaneous batch size per device = 64
  Total train batch size (w. parallel, distributed & accumulation) = 128
  Gradient Accumulation steps = 1
  Total optimization steps = 6690
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


RuntimeError: Caught RuntimeError in replica 1 on device 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/torch/nn/parallel/parallel_apply.py", line 61, in _worker
    output = module(*input, **kwargs)
  File "/usr/local/lib/python3.8/dist-packages/torch/nn/modules/module.py", line 1130, in _call_impl
    return forward_call(*input, **kwargs)
  File "/root/trevor/bert-on-mimic-iv/models/our_bert.py", line 1465, in forward
    outputs = self.bert(
  File "/usr/local/lib/python3.8/dist-packages/torch/nn/modules/module.py", line 1130, in _call_impl
    return forward_call(*input, **kwargs)
  File "/root/trevor/bert-on-mimic-iv/models/our_bert.py", line 1132, in forward
    encoder_outputs = self.encoder(
  File "/usr/local/lib/python3.8/dist-packages/torch/nn/modules/module.py", line 1130, in _call_impl
    return forward_call(*input, **kwargs)
  File "/root/trevor/bert-on-mimic-iv/models/our_bert.py", line 720, in forward
    layer_outputs = l

GPU can handle a batch up to 128 wrt memory and processing time. However, current processing code is too slow on the CPU to get a batch ready before the GPU can run it. 

```
Parameters which did not receive grad for rank 1: bert.encoder.layer.11.attention.output.dense.bias, bert.encoder.layer.11.attention.output.dense.weight, bert.encoder.layer.11.attention.self_attention.value.bias, bert.encoder.layer.11.attention.self_attention.value.weight, bert.encoder.layer.11.attention.self_attention.key.bias, bert.encoder.layer.11.attention.self_attention.key.weight, bert.encoder.layer.11.attention.self_attention.query.bias, bert.encoder.layer.11.attention.self_attention.query.weight,
```